# Parse ICD10 codes from CMS

In [1]:
# download ICD codes from CMS as an .xml file
# https://www.cms.gov/medicare/coding-billing/icd-10-codes/2024-icd-10-cm
import pandas as pd
import xml.dom.minidom
df = [] # results container
doc = xml.dom.minidom.parse('icd10cm_tabular_2024.xml')
# the .xml file has nested structure (chapter, section, diag, subdiag)
chapters = doc.documentElement.getElementsByTagName("chapter")
for chapter in chapters:
    sections = chapter.getElementsByTagName("section")
    for section in sections:
        diags = section.getElementsByTagName("diag")
        for diag in diags:
            diagNameElement = diag.getElementsByTagName("name")[0]
            diagDescElement = diag.getElementsByTagName("desc")[0]
            df.append([diagNameElement.childNodes[0].data, diagDescElement.childNodes[0].data])
            subdiags = diag.getElementsByTagName("diag")
            for subdiag in subdiags:
                subdiagNameElement = subdiag.getElementsByTagName("name")[0]
                subdiagDescElement = subdiag.getElementsByTagName("desc")[0]
                df.append([subdiagNameElement.childNodes[0].data, subdiagDescElement.childNodes[0].data])
df = pd.DataFrame(df, columns = ['code', 'name']) # convert to pandas df
df = df.drop_duplicates() # drop any duplicate rows
df['load_ts'] = pd.Timestamp.utcnow() # add timestamp
df.head(10) # quick check result
df.to_csv('icd10_codes_2024.csv', index = False) # save to .csv